# Bài học 01 - Giới thiệu về các Tác nhân AI

Chào mừng bạn đến với bài học đầu tiên trong khóa học **Tác nhân AI cho Người mới bắt đầu**!

Một **tác nhân AI** là một chương trình sử dụng mô hình ngôn ngữ lớn (LLM) làm động cơ suy luận và có thể thực hiện *hành động* trong thế giới thực — gọi API, truy vấn cơ sở dữ liệu hoặc chạy mã — để đạt được mục tiêu thay mặt người dùng.

Trong sổ tay này bạn sẽ xây dựng tác nhân đầu tiên của mình: một **Tác nhân Du lịch** đề xuất điểm đến nghỉ dưỡng. Trong quá trình đó bạn sẽ học cách:

1. Kết nối với Dịch vụ Microsoft Foundry Agent sử dụng **Khung Tác nhân Microsoft**.
2. Cung cấp cho tác nhân một **công cụ** — một hàm Python đơn giản mà nó có thể gọi.
3. Chạy tác nhân và kiểm tra phản hồi của nó.
4. Truyền phát phản hồi của tác nhân từng token một.


## Thiết lập

Trước khi chạy sổ tay này, hãy đảm bảo bạn đã:

1. **Một dự án Microsoft Foundry** với mô hình trò chuyện đã được triển khai (ví dụ: `gpt-4.1-mini`).
2. **Đăng nhập với Azure CLI** — chạy `az login` trong terminal của bạn.
3. **Đặt các biến môi trường cần thiết:**
   - `AZURE_AI_PROJECT_ENDPOINT` — điểm cuối dự án Microsoft Foundry của bạn.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — tên của mô hình đã triển khai của bạn.

Ô bên dưới sẽ cài đặt các gói Python mà bạn cần.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Tạo Agent Đầu Tiên Của Bạn

Một agent cần hai thứ:

- **Hướng dẫn** cho nó biết *nó là ai* và *cách hành xử* (một lời nhắc hệ thống).
- **Công cụ** — các hàm Python được trang trí với `@tool` mà agent có thể gọi để truy xuất thông tin hoặc thực hiện các hành động.

Dưới đây chúng ta định nghĩa một công cụ đơn giản trả về danh sách các điểm đến du lịch phổ biến. Agent sẽ sử dụng công cụ này khi người dùng hỏi về các đề xuất du lịch.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Phản hồi theo luồng

Để có trải nghiệm tương tác hơn, bạn có thể **truyền trực tiếp** phản hồi của tác nhân. Thay vì chờ đợi phản hồi đầy đủ, tác nhân sẽ xuất từng phần văn bản khi chúng được tạo ra. Điều này đặc biệt hữu ích trong các giao diện chat nơi bạn muốn hiển thị kết quả theo thời gian thực.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Tóm tắt

Trong bài học này, bạn đã học cách:

- **Tạo một provider** kết nối với Dịch vụ Microsoft Foundry Agent thông qua `FoundryChatClient`.
- **Định nghĩa một công cụ** sử dụng trình trang trí `@tool` để agent có thể gọi các hàm Python của bạn.
- **Chạy agent** với một tin nhắn từ người dùng và in phản hồi của nó.
- **Phát trực tiếp phản hồi** để xuất kết quả theo thời gian thực.

Trong bài học tiếp theo, chúng ta sẽ khám phá sâu hơn về các framework agentic và học cách cung cấp cho các agent công cụ mạnh mẽ hơn cùng khả năng lý luận đa bước.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
